##**Installing Dependencies**

In [1]:
pip install pytorch-msssim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 49.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

##**Importing Libraries**

In [2]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from pytorch_msssim import ssim
import numpy as np

##**Dataset**

**Dataset Preparation**

In [3]:
class SuperResolutionDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        """
        Args:
            lr_dir (str): Path to the low-resolution images folder.
            hr_dir (str): Path to the high-resolution images folder.
            transform (callable, optional): Transform to be applied on the images.
        """
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        # Assume filenames match in both folders.
        self.lr_files = sorted(os.listdir(lr_dir))
        self.hr_files = sorted(os.listdir(hr_dir))
        self.transform = transform

        if len(self.lr_files) != len(self.hr_files):
            raise ValueError("Mismatch in number of LR and HR images.")

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr_path = os.path.join(self.lr_dir, self.lr_files[idx])
        hr_path = os.path.join(self.hr_dir, self.hr_files[idx])

        # Check if the file is a NumPy array (.npy)
        if lr_path.endswith('.npy'):
            # If it's a NumPy array, load it using NumPy
            lr_image = np.load(lr_path)
            # Reshape the NumPy array if it's 3D; This line is likely the fix
            lr_image = lr_image.squeeze() # squeeze to remove dimensions of size 1
            # Convert the NumPy array to a PIL Image
            lr_image = Image.fromarray(lr_image.astype(np.uint8))  # Assuming uint8 data type
        else:
            # If it's a standard image file, open it using PIL
            lr_image = Image.open(lr_path).convert('L')

        # Similar check for the HR image
        if hr_path.endswith('.npy'):
            hr_image = np.load(hr_path)
            # Reshape the NumPy array if it's 3D; Same fix applied to HR image
            hr_image = hr_image.squeeze() # squeeze to remove dimensions of size 1
            hr_image = Image.fromarray(hr_image.astype(np.uint8))
        else:
            hr_image = Image.open(hr_path).convert('L')

        if self.transform:
            lr_image = self.transform(lr_image)
            hr_image = self.transform(hr_image)

        return lr_image, hr_image

**Data Augmentation**

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomCrop(64),  # Changed to 64
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [5]:
# For validation/testing, we use center crop and normalization.
test_transform = transforms.Compose([
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

**Defining Dataset Paths**

In [6]:
lr_dir = '/content/drive/MyDrive/Review papers/Dataset/LR'  # Folder with low-resolution images
hr_dir = '/content/drive/MyDrive/Review papers/Dataset/HR'

In [7]:
full_dataset = SuperResolutionDataset(lr_dir, hr_dir, transform=train_transform)

**Splitting Dataset**

In [8]:
train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

In [9]:
val_dataset = SuperResolutionDataset(lr_dir, hr_dir, transform=test_transform)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Training samples: 270
Testing samples: 30
Validation samples: 300


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


##**Modelling**

**Defining SRCNN model**

In [10]:
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        # First layer: large kernel to capture contextual information
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        # Second layer: non-linear mapping
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1)
        # Third layer: reconstruct the HR image
        self.conv3 = nn.Conv2d(32, 1, kernel_size=5, padding=2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SRCNN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

**Training the Model**


In [11]:
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)

        optimizer.zero_grad()
        outputs = model(lr_imgs)
        loss = criterion(outputs, hr_imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {avg_loss:.6f}")

Epoch 1/50, Training Loss: 0.179705
Epoch 2/50, Training Loss: 0.031160
Epoch 3/50, Training Loss: 0.010514
Epoch 4/50, Training Loss: 0.002603
Epoch 5/50, Training Loss: 0.001395
Epoch 6/50, Training Loss: 0.000928
Epoch 7/50, Training Loss: 0.000687
Epoch 8/50, Training Loss: 0.000548
Epoch 9/50, Training Loss: 0.000450
Epoch 10/50, Training Loss: 0.000371
Epoch 11/50, Training Loss: 0.000309
Epoch 12/50, Training Loss: 0.000258
Epoch 13/50, Training Loss: 0.000217
Epoch 14/50, Training Loss: 0.000186
Epoch 15/50, Training Loss: 0.000161
Epoch 16/50, Training Loss: 0.000142
Epoch 17/50, Training Loss: 0.000126
Epoch 18/50, Training Loss: 0.000113
Epoch 19/50, Training Loss: 0.000103
Epoch 20/50, Training Loss: 0.000093
Epoch 21/50, Training Loss: 0.000085
Epoch 22/50, Training Loss: 0.000077
Epoch 23/50, Training Loss: 0.000070
Epoch 24/50, Training Loss: 0.000064
Epoch 25/50, Training Loss: 0.000058
Epoch 26/50, Training Loss: 0.000054
Epoch 27/50, Training Loss: 0.000049
Epoch 28/5

**Evaluating the model**

In [12]:
model.eval()
total_mse = 0.0
total_psnr = 0.0
total_ssim = 0.0
num_batches = 0

with torch.no_grad():
    for lr_imgs, hr_imgs in test_loader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)
        outputs = model(lr_imgs)

        # Compute MSE for the batch
        mse_val = criterion(outputs, hr_imgs).item()
        total_mse += mse_val

        # Compute PSNR (assuming pixel values are in [0,1])
        psnr_val = 10 * math.log10(1 / mse_val) if mse_val > 0 else float('inf')
        total_psnr += psnr_val

        # Compute SSIM using pytorch_msssim; set data_range=1 for normalized images
        ssim_val = ssim(outputs, hr_imgs, data_range=1, size_average=True).item()
        total_ssim += ssim_val

        num_batches += 1

avg_mse = total_mse / num_batches
avg_psnr = total_psnr / num_batches
avg_ssim = total_ssim / num_batches

print("\nTest Evaluation Metrics:")
print(f"Average MSE: {avg_mse:.6f}")
print(f"Average PSNR: {avg_psnr:.2f} dB")
print(f"Average SSIM: {avg_ssim:.4f}")



Test Evaluation Metrics:
Average MSE: 0.000011
Average PSNR: 49.69 dB
Average SSIM: 0.9996


**Saving the model**


In [14]:
# Define the path where the model weights will be saved
model_path = "/content/scrnn_weights3b"

# Save only the model's state_dict (recommended)
torch.save(model.state_dict(), model_path)

print(f"Model weights saved to {model_path}")

Model weights saved to /content/scrnn_weights3b.pth
